In [14]:
import time
import pathlib
import tensorflow as tf
from tensorflow.keras import layers, applications, optimizers

# Enable mixed precision for hardware acceleration
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Ingest tf_flowers directly to ephemeral storage
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

batch_size = 32
img_height, img_width = 224, 224

# Provision datasets (80/20 split)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=123,
    image_size=(img_height, img_width), batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=123,
    image_size=(img_height, img_width), batch_size=batch_size
)

# Apply prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.


In [15]:
# Geometric augmentation mapped to GPU
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.2),
])

inputs = tf.keras.Input(shape=(img_height, img_width, 3))
x = data_augmentation(inputs)

# Native Caffe-style preprocessing
x = applications.resnet50.preprocess_input(x)

base_model = applications.ResNet50(
    weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)
)
base_model.trainable = False

# Forward pass with locked BN statistics
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

# Multi-class output (5 classes) forced to float32 for mixed precision stability
outputs = layers.Dense(5, activation='softmax', dtype='float32')(x)
model = tf.keras.Model(inputs, outputs)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy', # Updated for integer multi-class labels
    metrics=['accuracy']
)

print("\n--- Starting Feature Extraction ---")
start_fe = time.time()
history_fe = model.fit(train_ds, validation_data=val_ds, epochs=5)
fe_time = time.time() - start_fe


--- Starting Feature Extraction ---
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 31s 157ms/step - accuracy: 0.9894 - loss: 0.0322 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 1.0000 - loss: 1.5429e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - accuracy: 1.0000 - loss: 2.3143e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 1.0000 - loss: 1.9895e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 1.0000 - loss: 1.6241e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00


In [17]:
base_model.trainable = True

# Freeze all but the top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Starting Fine-Tuning ---")
start_ft = time.time()
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)
ft_time = time.time() - start_ft

# Benchmark Outputs
fe_acc = max(history_fe.history['val_accuracy'])
ft_acc = max(history_ft.history['val_accuracy'])

print(f"\nFeature Extraction -> Accuracy: {fe_acc:.4f} | Time: {fe_time:.1f} s")
print(f"Fine-Tuning        -> Accuracy: {ft_acc:.4f} | Time: {ft_time:.1f} s")


--- Starting Fine-Tuning ---
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 30s 167ms/step - accuracy: 1.0000 - loss: 2.5580e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 135ms/step - accuracy: 1.0000 - loss: 4.9129e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 13s 137ms/step - accuracy: 1.0000 - loss: 1.0151e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 13s 138ms/step - accuracy: 1.0000 - loss: 7.3085e-10 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 13s 140ms/step - accuracy: 1.0000 - loss: 1.0151e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00

Feature Extraction -> Accuracy: 1.0000 | Time: 70.7 s
Fine-Tuning        -> Accuracy: 1.0000 | Time: 80.3 s
